In [1]:
from dotenv import load_dotenv
load_dotenv()   # This reads the .env file in the current directory

True

In [2]:
import os
import time
import requests
import pandas as pd
import numpy as np
from sklearn.metrics import mean_squared_error, mean_absolute_error
from rouge_score import rouge_scorer

In [3]:
from dotenv import load_dotenv
load_dotenv()  # looks for .env in current or parent directory
import os
api_key = os.getenv("GROQ_API_KEY")

In [4]:
df = pd.read_csv("../outputs/unified_behavior_with_archetype.csv")
print(df.shape)
df.head()

(28294, 10)


,domain,user_id,item_id,review_text,rating,timestamp,archetype,cluster,avg_rating,dominant_value
0,yelp,mh_-eMZ6K5RLWhZyISBhwA,XQfwVwDr-v0ZS3_CbbE5Xw,"If you decide to eat here, just be aware it is...",3.0,2018-07-07 22:09:11,NaN,NaN,NaN,NaN
1,yelp,OyoGAe7OKpv6SyGZT5g77Q,7ATYjTIgM3jUlt4UM3IypQ,I've taken a lot of spin classes over the year...,5.0,2012-01-03 15:28:18,NaN,NaN,NaN,NaN
2,yelp,8g_iMtfSiwikVnbP2etR0A,YjUWPpI6HXG530lwP-fb2A,Family diner. Had the buffet. Eclectic assortm...,3.0,2014-02-05 20:30:30,NaN,NaN,NaN,NaN
3,yelp,_7bHUi9Uuf5__HHc_Q8guQ,kxX2SOes4o-D3ZQBkiMRfA,"Wow! Yummy, different, delicious. Our favo...",5.0,2015-01-04 00:01:03,NaN,NaN,NaN,NaN
4,yelp,bcjbaE6dDog4jkNY91ncLQ,e4Vwtrqf-wpJfwesgvdgxQ,Cute interior and owner (?) gave us tour of up...,4.0,2017-01-14 20:54:15,NaN,NaN,NaN,NaN


In [5]:
from sklearn.model_selection import train_test_split

train_list, test_list = [], []
for user, group in df.groupby('user_id'):
    if len(group) >= 2:
        train, test = train_test_split(group, test_size=0.2, random_state=42)
        train_list.append(train)
        test_list.append(test)
    else:
        train_list.append(group)

train_df = pd.concat(train_list, ignore_index=True)
test_df = pd.concat(test_list, ignore_index=True)
print(f"Train: {len(train_df)}, Test: {len(test_df)}")

Train: 25135, Test: 3159


In [6]:
# ===========================
#  Helper functions (copied/adapted from task_a.py)
# ===========================
def get_few_shot_examples(archetype, target_restaurant, unified_df, n=8):
    """Sample past reviews from same archetype."""
    archetype_reviews = unified_df[unified_df["archetype"] == archetype]
    if len(archetype_reviews) == 0:
        archetype_reviews = unified_df
    sample_size = min(n, len(archetype_reviews))
    sampled = archetype_reviews.sample(sample_size, random_state=42)
    examples = []
    for _, row in sampled.iterrows():
        examples.append({
            "rating": row["rating"],
            "review_text": row["review_text"][:200]
        })
    return examples

def build_prompt(persona, target_restaurant, context, few_shots):
    archetype = persona.get("archetype", "Balanced")
    dominant_value = persona.get("dominant_value", "neutral")
    avg_rating = persona.get("avg_rating", 3.5)
    
    context_tones = {
        "celebration": "Excited, generous",
        "sapa_budget": "Price‑sensitive, critical",
        "general": "Balanced and honest"
    }
    tone = context_tones.get(context, "Balanced and honest")
    
    prompt = f"""You are simulating a Nigerian user on a restaurant review platform.

**User Persona:**
- Archetype: {archetype}
- Dominant value: {dominant_value}
- Typical rating: {avg_rating:.1f} stars
- Current context: {context} → {tone}

**Few-shot examples of this user's past reviews:**
"""
    for i, ex in enumerate(few_shots, 1):
        prompt += f"{i}. Rating: {ex['rating']} stars\n   Review: {ex['review_text']}\n"
    
    prompt += f"""
**Now write a NEW review for this restaurant:**
Name: {target_restaurant.get('name', 'Unknown')}
Category: {target_restaurant.get('category', 'Nigerian')}
Price range: {target_restaurant.get('price_range', 'moderate')}
Location: {target_restaurant.get('location_type', 'Lagos')}
Description: {target_restaurant.get('description', 'A local spot')}

**Output exactly in this format:**
Rating: (1-5 integer)
Review: (natural language review in the user's voice, 50-150 words. Write in a natural mix of standard English and occasional Nigerian Pidgin/expressions – code‑switching.)

Do not add any extra text.
"""
    return prompt

def call_llm(prompt, api_key):
    url = "https://api.groq.com/openai/v1/chat/completions"
    headers = {"Authorization": f"Bearer {api_key}", "Content-Type": "application/json"}
    payload = {
        "model": "llama-3.3-70b-versatile",
        "messages": [{"role": "user", "content": prompt}],
        "temperature": 0.4,
        "max_tokens": 300
    }
    response = requests.post(url, headers=headers, json=payload)
    if response.status_code == 200:
        return response.json()["choices"][0]["message"]["content"]
    else:
        print(f"LLM error {response.status_code}: {response.text}")
        return None

def parse_response(response):
    lines = response.strip().split("\n")
    rating = 3
    review = "Could not parse."
    for line in lines:
        if line.lower().startswith("rating:"):
            try:
                rating = int(line.split(":")[1].strip())
                rating = max(1, min(5, rating))
            except:
                rating = 3
        elif line.lower().startswith("review:"):
            review = line.split(":", 1)[1].strip()
    return rating, review

In [7]:
from dotenv import load_dotenv
load_dotenv()   # Load .env file from project root

import os
import time
import pandas as pd
import numpy as np
from sklearn.metrics import mean_squared_error, mean_absolute_error
from rouge_score import rouge_scorer

API_KEY = os.getenv("GROQ_API_KEY")
if not API_KEY:
    raise ValueError("Set GROQ_API_KEY in .env file or environment variable")

sample_size = 30
eval_df = test_df.sample(min(sample_size, len(test_df)), random_state=42)

# Compute user average rating from training data
user_avg_rating = train_df.groupby('user_id')['rating'].mean().to_dict()

# Baseline on the same eval_df
eval_baseline_preds = eval_df['user_id'].map(user_avg_rating).fillna(3.0)
eval_baseline_rmse = np.sqrt(mean_squared_error(eval_df['rating'], eval_baseline_preds))
eval_baseline_mae = mean_absolute_error(eval_df['rating'], eval_baseline_preds)
print(f"Baseline on eval subset: RMSE {eval_baseline_rmse:.4f}, MAE {eval_baseline_mae:.4f}")

pred_ratings = []
true_ratings = []
generated_reviews = []
true_reviews = []

for i, (idx, row) in enumerate(eval_df.iterrows()):
    user_id = row['user_id']
    true_rating = row['rating']
    true_review = row['review_text']
    
    # Get persona from train_df
    user_train = train_df[train_df['user_id'] == user_id]
    if len(user_train) > 0:
        persona = user_train.iloc[0]
    else:
        # fallback: random persona from training
        persona = train_df.sample(1, random_state=42).iloc[0]
    
    # Build target (minimal)
    target = {
        'name': row.get('item_id', 'Unknown'),
        'category': 'Restaurant',
        'price_range': 'Moderate',
        'location_type': 'Lagos',
        'description': row['review_text'][:200]
    }
    
    few_shots = get_few_shot_examples(persona['archetype'], target, train_df, n=5)
    prompt = build_prompt(persona, target, context="general", few_shots=few_shots)
    
    output = call_llm(prompt, API_KEY)   # call_llm uses the key
    if output:
        rating, review = parse_response(output)
        pred_ratings.append(rating)
        true_ratings.append(true_rating)
        generated_reviews.append(review)
        true_reviews.append(true_review)
        print(f" Processed {i+1}/{len(eval_df)}")
    else:
        print(f" Failed at {idx}")
    
    time.sleep(2)   # rate limit

print("Evaluation done.")

Baseline on eval subset: RMSE 1.1212, MAE 0.7242
 Processed 1/30
 Processed 2/30
 Processed 3/30
 Processed 4/30
 Processed 5/30
 Processed 6/30
 Processed 7/30
 Processed 8/30
 Processed 9/30
 Processed 10/30
 Processed 11/30
 Processed 12/30
 Processed 13/30
 Processed 14/30
 Processed 15/30
 Processed 16/30
LLM error 429: {"error":{"message":"Rate limit reached for model `llama-3.3-70b-versatile` in organization `org_01ksbe519ffqys5zm8z7kkc3hz` service tier `on_demand` on tokens per day (TPD): Limit 100000, Used 99745, Requested 800. Please try again in 7m50.88s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing","type":"tokens","code":"rate_limit_exceeded"}}

 Failed at 457
LLM error 429: {"error":{"message":"Rate limit reached for model `llama-3.3-70b-versatile` in organization `org_01ksbe519ffqys5zm8z7kkc3hz` service tier `on_demand` on tokens per day (TPD): Limit 100000, Used 99742, Requested 658. Please try again in 5m45.6s. Need more toke

In [8]:
if pred_ratings:
    rmse = np.sqrt(mean_squared_error(true_ratings, pred_ratings))
    mae = mean_absolute_error(true_ratings, pred_ratings)
    print(f"Agent RMSE: {rmse:.4f}")
    print(f"Agent MAE: {mae:.4f}")
    
    # ROUGE-L
    scorer = rouge_scorer.RougeScorer(['rougeL'], use_stemmer=True)
    rouge_scores = []
    for gen, ref in zip(generated_reviews, true_reviews):
        scores = scorer.score(ref, gen)
        rouge_scores.append(scores['rougeL'].fmeasure)
    print(f"Agent ROUGE-L F1: {np.mean(rouge_scores):.4f}")
else:
    print("No predictions were generated.")

Agent RMSE: 1.2437
Agent MAE: 0.7188
Agent ROUGE-L F1: 0.1405


In [11]:
if pred_ratings:
    rmse = np.sqrt(mean_squared_error(true_ratings, pred_ratings))
    mae = mean_absolute_error(true_ratings, pred_ratings)
    print(f"Improved Agent RMSE: {rmse:.4f}")
    print(f"Improved Agent MAE: {mae:.4f}")
    
    scorer = rouge_scorer.RougeScorer(['rougeL'], use_stemmer=True)
    rouge_scores = [scorer.score(ref, gen)['rougeL'].fmeasure 
                    for gen, ref in zip(generated_reviews, true_reviews)]
    print(f"Improved Agent ROUGE-L F1: {np.mean(rouge_scores):.4f}")

Improved Agent RMSE: 1.2437
Improved Agent MAE: 0.7188
Improved Agent ROUGE-L F1: 0.1405


In [9]:
# Compute ROUGE scores
scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)

rouge1_scores = []
rouge2_scores = []
rougeL_scores = []

for gen, ref in zip(generated_reviews, true_reviews):
    scores = scorer.score(ref, gen)          # returns dict: {'rouge1': Score, 'rouge2': Score, 'rougeL': Score}
    rouge1_scores.append(scores['rouge1'].fmeasure)
    rouge2_scores.append(scores['rouge2'].fmeasure)
    rougeL_scores.append(scores['rougeL'].fmeasure)

avg_rouge1 = sum(rouge1_scores) / len(rouge1_scores)
avg_rouge2 = sum(rouge2_scores) / len(rouge2_scores)
avg_rougeL = sum(rougeL_scores) / len(rougeL_scores)

print(f"ROUGE-1 F1: {avg_rouge1:.4f}")
print(f"ROUGE-2 F1: {avg_rouge2:.4f}")
print(f"ROUGE-L F1: {avg_rougeL:.4f}")

ROUGE-1 F1: 0.1955
ROUGE-2 F1: 0.0534
ROUGE-L F1: 0.1405


In [10]:
# User‑mean baseline: predict each user's average rating from training
user_avg_rating = train_df.groupby('user_id')['rating'].mean().to_dict()

# Apply to test set
baseline_preds = test_df['user_id'].map(user_avg_rating).fillna(3.0)  # fallback to 3.0 for new users

# Compute metrics
baseline_rmse = np.sqrt(mean_squared_error(test_df['rating'], baseline_preds))
baseline_mae = mean_absolute_error(test_df['rating'], baseline_preds)

print(f"User‑mean baseline RMSE: {baseline_rmse:.4f}")
print(f"User‑mean baseline MAE: {baseline_mae:.4f}")

User‑mean baseline RMSE: 0.8147
User‑mean baseline MAE: 0.5095
